## Measure exploration

In this section i analyze the measure by category and dimensions. it will help me to understand the importance of different categories.

In [2]:
import sqlite3
from pathlib import Path
import pandas as pd

conn = sqlite3.connect(Path.cwd().parent/"database/ue_market.db")
pd.read_sql("SELECT name FROM sqlite_master WHERE   type= 'table'", conn)

,name
0,trade_category
1,trade_total
2,unemployment_ue
3,data_breaks_ue
4,gdp_ue


In [3]:
pd.read_sql("""
            SELECT 
                country,
                ROUND(AVG(CAST(unemployment_rate AS FLOAT)), 2) AS AVG_unemployment_by_country,
                MIN(unemployment_rate) AS Min_unemployment,
                MAX(unemployment_rate) AS Max_unemployment,
                MAX(CASE WHEN unemployment_rate = max_in_country THEN year END) AS year_of_max,
                MAX(CASE WHEN unemployment_rate = min_in_country THEN year END) AS year_of_min
            FROM (
                    SELECT country, year, unemployment_rate,
                    MAX(unemployment_rate) OVER (PARTITION BY country) AS max_in_country,
                    MIN(unemployment_rate) OVER (PARTITION BY country) AS  min_in_country
                    FROM unemployment_ue
                )
            GROUP BY country
            ORDER BY AVG_unemployment_by_country DESC
            """,
            conn)

,country,AVG_unemployment_by_country,Min_unemployment,Max_unemployment,year_of_max,year_of_min
0,Greece,15.82,8.9,23.9,2016,2025
1,Spain,14.37,10.5,19.6,2016,2025
2,Italy,9.07,6.1,11.7,2016,2025
3,France,8.25,7.3,10.1,2016,2023
4,Finland,7.94,6.8,9.7,2025,2022
5,Sweden,7.71,6.5,8.9,2021,2018
6,Croatia,7.67,4.9,13.0,2016,2025
7,Cyprus,7.58,4.4,13.0,2016,2025
8,Latvia,7.50,6.3,9.7,2016,2019
9,Portugal,7.35,6.0,11.5,2016,2025


In [4]:
pd.read_sql(""" 
    SELECT 
        country,
        ROUND(AVG(CAST(gdp_meur AS FLOAT)),2) AS avg_gdp_meur,
        MIN(gdp_meur) AS min_gdp_meur,
        MAX(gdp_meur) AS max_gdp_meur,
        MAX(CASE WHEN gdp_meur = max_in_country THEN year END) AS year_of_max,
        MIN(CASE WHEN gdp_meur = min_in_country THEN year END) AS year_of_min
    FROM
        (SELECT
            country, year, gdp_meur,
            MAX(gdp_meur) OVER (PARTITION BY country) AS max_in_country,
            MIN(gdp_meur) OVER (PARTITION BY country) AS min_in_country
        FROM gdp_ue
        WHERE unit = 'real')
    GROUP BY country
    ORDER BY avg_gdp_meur DESC""",
    conn)

,country,avg_gdp_meur,min_gdp_meur,max_gdp_meur,year_of_max,year_of_min
0,Germany,3020404.45,2906299.4,3094378.2,2022,2016
1,France,2240696.61,2082756.5,2377855.8,2025,2020
2,Italy,1640055.61,1483849.6,1732624.6,2025,2020
3,Spain,1188408.92,1065944.8,1318439.0,2025,2020
4,Netherlands,752206.65,688532.8,812190.6,2025,2016
5,Poland,520948.44,434111.0,599853.0,2025,2016
6,Sweden,454126.27,421981.0,484652.3,2025,2016
7,Belgium,417458.35,388931.6,449430.7,2025,2016
8,Ireland,340053.78,238676.5,446967.6,2025,2016
9,Austria,334710.88,316374.9,349646.5,2022,2020


In [5]:
pd.read_sql("""
WITH cte_export AS (
    SELECT country_code, SUM(value_meur) AS total_export
    FROM trade_total WHERE indic_et = 'export'
    GROUP BY country_code
),
cte_import AS (
    SELECT country_code, SUM(value_meur) AS total_import
    FROM trade_total WHERE indic_et = 'import'
    GROUP BY country_code
),
CTE_maxmin_export AS(
	SELECT
			country_code,
			MAX(value_meur) AS max_export,
			MIN(value_meur) AS min_export,
			MAX(CASE WHEN value_meur = max_export THEN year END) AS max_export_year,
			MIN(CASE WHEN value_meur = min_export THEN year END) AS min_export_year
		FROM (
			SELECT
				country_code,
				year,
				value_meur,
				MAX(value_meur) OVER (PARTITION BY country_code) AS max_export,
				MIN(value_meur) OVER (PARTITION BY country_code) AS min_export
			FROM trade_total
			WHERE indic_et = 'export'
		)f
		GROUP BY country_code)
,CTE_maxmin_import AS(
	SELECT
			country_code,
			MAX(value_meur) AS max_import,
			MIN(value_meur) AS min_import,
			MAX(CASE WHEN value_meur = max_import THEN year END) AS max_import_year,
			MIN(CASE WHEN value_meur = min_import THEN year END) AS min_import_year
		FROM (
			SELECT
				country_code,
				year,
				value_meur,
				MAX(value_meur) OVER (PARTITION BY country_code) AS max_import,
				MIN(value_meur) OVER (PARTITION BY country_code) AS min_import
			FROM trade_total
			WHERE indic_et = 'import'
		)f
		GROUP BY country_code),
CTE_balance AS(
	SELECT 
	country_code,
	ROUND(AVG(CAST(value_meur AS FLOAT)),1) AS AVG_balance
FROM trade_total
WHERE indic_et = 'balance'
GROUP BY country_code)
,CTE_country_names AS(
SELECT DISTINCT
	country_code,
	country
	FROM unemployment_ue)
		
SELECT 
    e.country_code,
	n.country,
    e.total_export,
    i.total_import,
	me.max_export,
	me.max_export_year,
	me.min_export,
	me.min_export_year,
	mi.max_import,
	mi.max_import_year,
	mi.min_import,
	mi.min_import_year,
	ba.AVG_balance
FROM cte_export e
JOIN cte_import i ON e.country_code = i.country_code
JOIN CTE_maxmin_export me ON e.country_code = me.country_code
JOIN CTE_maxmin_import mi ON e.country_code = mi.country_code
JOIN CTE_balance ba ON e.country_code = ba.country_code
JOIN CTE_country_names n ON e.country_code = n.country_code
ORDER BY AVG_balance DESC
""", conn)

,country_code,country,total_export,total_import,max_export,max_export_year,min_export,min_export_year,max_import,max_import_year,min_import,min_import_year,AVG_balance
0,DE,Germany,26972337.8,22481136.5,1593593.0,2022,651259.2,2002,1505973.1,2022,518488.2,2002,187133.4
1,NL,Netherlands,12796187.6,11439151.4,917539.9,2022,258099.0,2002,853919.7,2022,231878.7,2002,56543.2
2,IE,Ireland,2982684.6,1853548.7,258934.7,2025,81997.3,2003,142770.4,2025,44955.5,2009,47047.3
3,IT,Italy,10067852.7,9654343.5,643153.3,2025,264615.6,2003,660210.0,2022,261225.9,2002,17229.5
4,BE,Belgium,8757281.3,8395786.5,607029.0,2022,225969.6,2003,594550.0,2022,207695.2,2003,15062.3
5,CZ,Czechia,3261416.2,3005145.9,251327.3,2025,40706.3,2002,225184.9,2022,42995.3,2002,10677.9
6,DK,Denmark,2105599.6,1912609.1,131664.6,2025,58798.1,2003,122783.5,2022,50767.8,2003,8041.3
7,SE,Sweden,3180914.0,3030588.5,187838.7,2022,86187.7,2002,193067.6,2022,70806.8,2002,6263.6
8,HU,Hungary,2152365.1,2090684.4,153974.0,2023,36503.1,2002,156094.3,2022,39926.9,2002,2570.0
9,FI,Finland,1441226.6,1428136.4,81884.7,2022,45063.4,2009,92469.4,2022,36186.5,2002,545.4


In [12]:
pd.read_sql("""
WITH CTE_ranked AS(
SELECT 
        country_code,
        product_description,
        SUM(value_meur) AS total_export,
        ROW_NUMBER() OVER (PARTITION BY country_code ORDER BY SUM(value_meur) DESC) AS rk
    FROM trade_category
    WHERE indic_et = 'export'
    GROUP BY country_code, product_description),
CTE_country_name AS(
SELECT
		country_code,
		country
		FROM unemployment_ue)
	
SELECT
    a.country_code,
	n.country,
    MAX(CASE WHEN rk=1 THEN product_description END) AS first_product,
    MAX(CASE WHEN rk=1 THEN total_export END) AS first_product_export,
    MAX(CASE WHEN rk=2 THEN product_description END) AS second_product,
    MAX(CASE WHEN rk=2 THEN total_export END) AS second_product_export,
    MAX(CASE WHEN rk=3 THEN product_description END) AS third_product,
    MAX(CASE WHEN rk=3 THEN total_export END) AS third_product_export,
    MAX(CASE WHEN rk=1 THEN total_export END) + 
            MAX(CASE WHEN rk=2 THEN total_export END) +
                MAX(CASE WHEN rk=3 THEN total_export END) AS total_best_3
FROM CTE_ranked a
JOIN CTE_country_name n on a.country_code =  n.country_code
WHERE rk <= 3
GROUP BY a.country_code, n.country
ORDER BY first_product_export DESC
""", 
conn)

,country_code,country,first_product,first_product_export,second_product,second_product_export,third_product,third_product_export,total_best_3
0,DE,Germany,machinery_transport,13193439.4,other_manufactured,6399883.9,chemicals,4395974.1,23989297.4
1,FR,France,machinery_transport,4146330.1,other_manufactured,2506682.2,chemicals,2014734.9,8667747.2
2,NL,Netherlands,machinery_transport,4032448.4,other_manufactured,2502306.9,chemicals,2138210.9,8672966.2
3,IT,Italy,other_manufactured,3635849.2,machinery_transport,3548773.9,chemicals,1350752.2,8535375.3
4,BE,Belgium,chemicals,2648710.2,other_manufactured,2184639.3,machinery_transport,1935943.9,6769293.4
5,ES,Spain,machinery_transport,1959956.4,other_manufactured,1522422.4,food_drinks_tobacco,894123.0,4376501.8
6,CZ,Czechia,machinery_transport,1832574.7,other_manufactured,902933.7,chemicals,212659.5,2948167.9
7,IE,Ireland,chemicals,1743626.9,machinery_transport,500350.7,other_manufactured,374836.0,2618813.6
8,PL,Poland,machinery_transport,1643159.9,other_manufactured,1502605.4,food_drinks_tobacco,539833.5,3685598.8
9,AT,Austria,machinery_transport,1318150.3,other_manufactured,1087642.8,chemicals,440544.4,2846337.5


This exploratory analysis has provided an overview of trade volumes, unemployment levels, and export composition across EU countries. Several significant trends emerge: Germany clearly dominates EU trade, and a distinct divide is evident between net exporters (Germany, the Netherlands, Italy) and net importers (France, Spain, Greece). <br>
Furthermore,the top three categories per country appear to account for the largest share of exports; across countries, these are most often machinery and transport equipment, chemicals, and miscellaneous manufactured goods, with food/beverages/tobacco also frequent.. We will quantify these shares deply in subsequent analyses. <br>
Unemployment is structurally higher in Greece and Spain. GDP figures show growth over the 2016–2025 period, as would be expected for an indicator that tends to rise year on year. These findings will guide the analysis addressing the three research questions, during which I will integrate the datasets to examine the relationships between trade, GDP, and unemployment.